# Phase 2: model results and daily demand types

This notebook reads the saved reports and uses package charts. It performs no downloads or training. Run from the repository root or its notebooks folder. Prepare and train with `scripts/prepare_phase2.py` and `scripts/train_models.py` when regenerating the artifacts.


In [ ]:
import pandas as pd
from IPython.display import display

from spain_electricity import charts
from spain_electricity.config import REPO_ROOT, DataPaths
from spain_electricity.model_results import comparison_table, saved_reports

paths = DataPaths(REPO_ROOT / "data")
reports = saved_reports(paths)
display(comparison_table(reports))

## Interpretation

Train: 2021-2024. Parameter selection: 2025. Test: 2026. The price and renewable models refit the selected configuration on train plus paired validation rows. REE is the demand baseline. Price and renewable share use the exact elapsed UTC t-24h value. All comparisons use paired rows. The compact sample does not establish full-year performance. Negative electricity prices and renewable shares above one are valid.


In [ ]:
for name in ("demand", "price", "renewable_share"):
    report = reports.get(name, {})
    if report.get("features"):
        print(name)
        display(charts.importance_chart(pd.DataFrame(report["features"]), charts.LIGHT))

## Feature weights

Tree importances and standardized linear coefficients have different interpretations. Correlated predictors may share importance. Neither these weights nor the contemporaneous estimates establish causality or day-ahead forecasting skill.

## KMeans day types

KMeans selects k using training-day silhouette, then assigns other complete days to those centers. Each profile has mean one. Spring hour 2 is interpolated only on the known DST day, and the repeated autumn hour is averaged. Arbitrary cluster IDs describe demand shapes, without automatic weather labels.


In [ ]:
if "day_types" in reports:
    report = reports["day_types"]
    display(pd.DataFrame([{"k": k, "silhouette": value} for k, value in report["silhouettes"].items()]))
    summary = pd.DataFrame(report["summary"])
    display(charts.profiles_chart(summary, charts.LIGHT))
    display(summary.drop(columns="mean_profile"))

## Next validation

Review the saved row counts and units, compare each target against its baseline, and inspect errors before explaining their causes. Full-history evaluation remains required for the final delivery. Phase 3 is outside this notebook.
